# Make Model Figure - FiPy Output

Read FiPy VTK file and create model figure

This notebook is designed for FiPy model output, which uses a single VTK file
containing both cell variables and boundary fluxes in cell-centered form.

**Author:** Elco Luijendijk

In [15]:
import matplotlib
matplotlib.use('Agg')

import matplotlib.patches
import matplotlib.collections

import sys
import os
import itertools
import random

import numpy as np
import matplotlib.pyplot as pl
import scipy.interpolate
import vtk

from lib.grompy_lib import get_normal_flux_to_bnd

from model_input.figure_options import *

from matplotlib import ticker

# VTU file reader using VTK library (handles binary appended data)
def read_vtu_file_vtk(filename):
    """Read VTU file using VTK library"""
    
    reader = vtk.vtkXMLUnstructuredGridReader()
    reader.SetFileName(filename)
    reader.Update()
    
    output = reader.GetOutput()
    
    # Get points
    points_vtk = output.GetPoints()
    n_points = points_vtk.GetNumberOfPoints()
    xy_pts = np.zeros((n_points, 3))
    for i in range(n_points):
        p = points_vtk.GetPoint(i)
        xy_pts[i] = p
    
    # Get cells
    cells = output.GetCells()
    n_cells = cells.GetNumberOfCells()
    
    conn = []
    for i in range(n_cells):
        cell = output.GetCell(i)
        n_pts = cell.GetNumberOfPoints()
        pts = [cell.GetPointId(j) for j in range(n_pts)]
        conn.append(pts)
    
    conn = np.array(conn, dtype=object)
    
    # Get cell data
    cell_data = output.GetCellData()
    n_arrays = cell_data.GetNumberOfArrays()
    
    cell_var_names = []
    cell_var_arrays = []
    
    for i in range(n_arrays):
        array = cell_data.GetArray(i)
        name = array.GetName()
        n_comp = array.GetNumberOfComponents()
        n_tuples = array.GetNumberOfTuples()
        
        cell_var_names.append(name)
        
        # Convert to numpy
        data = np.zeros((n_tuples, n_comp))
        for j in range(n_tuples):
            if n_comp == 1:
                data[j, 0] = array.GetValue(j)
            else:
                for k in range(n_comp):
                    data[j, k] = array.GetComponent(j, k)
        
        if n_comp == 1:
            data = data.ravel()
        
        cell_var_arrays.append(data)
    
    # Get point data
    point_data = output.GetPointData()
    n_arrays = point_data.GetNumberOfArrays()
    
    pt_var_names = []
    pt_var_arrays = []
    
    for i in range(n_arrays):
        array = point_data.GetArray(i)
        name = array.GetName()
        n_comp = array.GetNumberOfComponents()
        n_tuples = array.GetNumberOfTuples()
        
        pt_var_names.append(name)
        
        # Convert to numpy
        data = np.zeros((n_tuples, n_comp))
        for j in range(n_tuples):
            if n_comp == 1:
                data[j, 0] = array.GetValue(j)
            else:
                for k in range(n_comp):
                    data[j, k] = array.GetComponent(j, k)
        
        if n_comp == 1:
            data = data.ravel()
        
        pt_var_arrays.append(data)
    
    return xy_pts, conn, pt_var_names, pt_var_arrays, cell_var_names, cell_var_arrays

## Figure Options

In [ ]:
# For direct file input, uncomment and set the path:
# vtk_file = '/path/to/your/file.vtu'
#vtk_file = "model_output/median_case/vtk_files/runS0_L_8045.0_final_output.vtu"

#vtk_file = "model_output/salt_wedge_benchmark/vtk_files/runS0_specified_pressure_[0, 68.569]_final_output.vtu"

vtk_file = "model_output/salt_wedge_benchmark/vtk_files/runS1_specified_pressure_[0, 19.591]_final_output.vtu"

# only plot concentration or plot all model results
conc_only = False

## Helper Functions

In [17]:
def add_subplot_axes(ax, rect, axisbg='w'):
    """
    embed a new subplot in existing subplot

    based on:
    http://stackoverflow.com/questions/17458580/embedding-small-plots-inside-subplots-in-matplotlib
    """

    fig = pl.gcf()
    box = ax.get_position()

    width = box.width
    height = box.height

    inax_position = ax.transAxes.transform(rect[0:2])
    transFigure = fig.transFigure.inverted()
    infig_position = transFigure.transform(inax_position)

    x = infig_position[0]
    y = infig_position[1]

    width *= rect[2]
    height *= rect[3]

    subax = fig.add_axes([x, y, width, height])

    x_labelsize = subax.get_xticklabels()[0].get_size()
    y_labelsize = subax.get_yticklabels()[0].get_size()

    x_labelsize *= rect[2] ** 0.5
    y_labelsize *= rect[3] ** 0.5

    subax.xaxis.set_tick_params(labelsize=x_labelsize)
    subax.yaxis.set_tick_params(labelsize=y_labelsize)

    return subax

In [18]:
def convert_to_grid(x, y, qx, qy, dx=1.0, dy=1.0):

    """
    interpolate a variable to a regular raster

    """

    # interpolate pressure on regular grid
    xi = np.arange(x.min(), x.max() + dx, dx)
    yi = np.arange(y.min(), y.max() + dy, dy)
    xg, yg = np.meshgrid(xi, yi)
    xgqf, ygqf = xg.flatten(), yg.flatten()

    # interpolate u to grid
    xyq_pts = np.vstack((x, y)).T
    qx_interp = scipy.interpolate.griddata(xyq_pts,
                                           qx,
                                           np.vstack((xgqf, ygqf)).T,
                                           method='linear')
    qy_interp = scipy.interpolate.griddata(xyq_pts,
                                           qy,
                                           np.vstack((xgqf, ygqf)).T,
                                           method='linear')

    qx_interp1 = np.resize(qx_interp, xg.shape)
    qy_interp1 = np.resize(qy_interp, xg.shape)

    return xg, yg, qx_interp1, qy_interp1

## File Setup and Configuration

In [19]:
# Handle VTK file setup - simplified for single file
if vtk_file is not None and 'vtu' in vtk_file:
    try:
        assert vtk_file[-4:] == '.vtu'
    except AssertionError:
        raise NameError('file name does not end with .vtu, are you sure this is a grompy VTK file?')

    vtk_files = [vtk_file]
    folder = os.path.split(vtk_file)[0]
else:
    raise ValueError('Please specify a valid VTK file')

print(f'Using VTK file: {vtk_file}')

Using VTK file: model_output/salt_wedge_benchmark/vtk_files/runS1_specified_pressure_[0, 19.591]_final_output.vtu


In [20]:
# Create output directory
folder_base = os.path.split(folder)[0]

fig_folder = os.path.join(folder_base, 'fig')
path = os.path.normpath(folder_base)
b = path.split(os.sep)

if os.path.exists(fig_folder) is False:
    os.mkdir(fig_folder)

print(f'Output directory: {fig_folder}')

Output directory: model_output/salt_wedge_benchmark/fig


## Main Processing

In [21]:
# Process VTK file
for vtk_file_path in vtk_files[::-1]:

    fn = vtk_file_path

    print(f'Reading VTK file: {fn}')

    xy_pts, conn, pt_var_names, pt_var_arrays, cell_var_names, cell_var_arrays = \
        read_vtu_file_vtk(fn)

    vtk_filename = os.path.split(vtk_file_path)[-1]

    # construct label from filename
    f = vtk_filename.split('.')[:-1]
    f = ''.join(f)
    f1 = f.split('_')
    
    # Find where 'output' or 'final' appears in filename
    plot_title = None
    if add_title is True:
        # Create simplified title from filename
        if 'final' in f1:
            ind = f1.index('final')
            f2 = f1[:ind]
        else:
            f2 = f1
        
        plot_title = ' '.join(f2)

    # get fluxes
    qx_ind = cell_var_names.index('velocity_x')
    qx = cell_var_arrays[qx_ind]
    qy_ind = cell_var_names.index('velocity_y')
    qy = cell_var_arrays[qy_ind]

    # make pts 2d
    xy_pts = xy_pts[:, :2]

    # make closed form of polygons
    conn_cl = np.zeros((conn.shape[0], conn.shape[1] + 1), dtype=int)
    conn_cl[:, :conn.shape[1]] = conn
    conn_cl[:, -1] = conn[:, 0]

    print('Finished reading files')

Reading VTK file: model_output/salt_wedge_benchmark/vtk_files/runS1_specified_pressure_[0, 19.591]_final_output.vtu
Finished reading files


In [22]:
# Extract boundary fluxes from cell variables
print('Finding boundary fluxes')

# Look for boundary flux variables - they're now in cell_var_names
flux_surface_norm_x = None
flux_surface_norm_y = None

if 'flux_surface_norm_x' in cell_var_names:
    x_ind = cell_var_names.index('flux_surface_norm_x')
    flux_surface_norm_x = cell_var_arrays[x_ind]
    
if 'flux_surface_norm_y' in cell_var_names:
    y_ind = cell_var_names.index('flux_surface_norm_y')
    flux_surface_norm_y = cell_var_arrays[y_ind]

if flux_surface_norm_x is None or flux_surface_norm_y is None:
    print('Warning: boundary flux variables not found in VTK file')
    print(f'Available cell variables: {cell_var_names}')
    # Use nodal flux as fallback if available
    if 'nodal_flux' in cell_var_names:
        nf_ind = cell_var_names.index('nodal_flux')
        nodal_flux = cell_var_arrays[nf_ind]
        if nodal_flux.ndim == 2 and nodal_flux.shape[1] == 2:
            flux_surface_norm_x = nodal_flux[:, 0]
            flux_surface_norm_y = nodal_flux[:, 1]
else:
    print('Successfully found boundary flux variables')

# Look for surface mask
if 'surface_mask' in cell_var_names:
    sm_ind = cell_var_names.index('surface_mask')
    surface_mask = cell_var_arrays[sm_ind]
    bnd_ind = surface_mask > 0
else:
    print('Warning: surface_mask not found, using all cells')
    bnd_ind = np.ones(len(cell_var_arrays[0]), dtype=bool)

print(f'Found {bnd_ind.sum()} boundary cells')

Finding boundary fluxes
Successfully found boundary flux variables
Found 212 boundary cells


In [23]:
# Calculate element centers and create polygons
print('Generating polygons for elements')
polys = [xy_pts[conn_i.astype(int)] for conn_i in conn]
patches = [matplotlib.patches.Polygon(poly) for poly in polys]

# calculate polygon centre
polys_array = np.array(polys)

element_x = np.mean(polys_array[:, :, 0], axis=1)
element_y = np.mean(polys_array[:, :, 1], axis=1)

print('Element centers calculated')

Generating polygons for elements
Element centers calculated


In [24]:
conn[0]

array([214, 213, 1], dtype=object)

In [25]:
# Process boundary coordinates and fluxes
if bnd_ind.sum() > 0:
    topxy = element_x[bnd_ind], element_y[bnd_ind]
    topxy = np.vstack(topxy).T
    
    x_order = np.argsort(topxy[:, 0])
    top_xy_sorted = topxy[x_order]
    
    # Get boundary fluxes for boundary cells
    if flux_surface_norm_y is not None:
        bnd_flux_y = flux_surface_norm_y[bnd_ind][x_order]
        bnd_flux_y *= year  # Convert to m/yr
        
        # Create normalized flux for plotting
        nodal_flux_corrected = np.vstack((np.zeros_like(bnd_flux_y), bnd_flux_y)).T
    else:
        print('Warning: could not extract boundary flux y-component')
        nodal_flux_corrected = np.zeros((top_xy_sorted.shape[0], 2))
    
    # Get hydraulic head at boundary if available
    h_top = None
    if 'hydraulic_head' in cell_var_names:
        h_ind = cell_var_names.index('hydraulic_head')
        h = cell_var_arrays[h_ind]
        h_top = h[bnd_ind][x_order]
else:
    print('No boundary cells found')
    top_xy_sorted = np.array([[0, 0]])
    nodal_flux_corrected = np.zeros((1, 2))
    h_top = None

In [26]:
# Create figure
print('Creating figures')

fig, axs = pl.subplots(2, 1, gridspec_kw={'height_ratios': [1, 3], 'hspace':0.0}, 
                       sharex=True, figsize=(8,4), layout="constrained")
tax, ax = axs

for axi in axs:
    axi.spines['top'].set_visible(False)
    axi.spines['right'].set_visible(False)
    axi.get_xaxis().tick_bottom()
    axi.get_yaxis().tick_left()

cax2 = None
if add_colorbar is True:
    rect1 = [0.7, 0.05, 0.3, 0.25]
    cbbox = ax.inset_axes(rect1)
    [cbbox.spines[k].set_visible(False) for k in cbbox.spines]
    cbbox.tick_params(
        axis = 'both',
        left = False,
        top = False,
        right = False,
        bottom = False,
        labelleft = False,
        labeltop = False,
        labelright = False,
        labelbottom = False
    )

    cbbox.set_facecolor([1,1,1,0.7])
    cax2 = cbbox.inset_axes([0.1, 0.8, 0.8, 0.15])

print('Figure created')

Creating figures
Figure created


In [27]:
# Prepare visualization data
if conc_only is True:
    c_ind = None
    if 'concentration' in cell_var_names:
        c_ind = cell_var_names.index('concentration')
    elif 'conc' in cell_var_names:
        c_ind = cell_var_names.index('conc')

    if c_ind is None:
        print('Warning: concentration variable not found, plotting all variables')
        cell_var_arrays_plot = cell_var_arrays
        cell_var_names_plot = cell_var_names
    else:
        cell_var_arrays_plot = [cell_var_arrays[c_ind]]
        cell_var_arrays_plot[0] = np.round(cell_var_arrays_plot[0], decimals=3)
        cell_var_names_plot = [cell_var_names[c_ind]]
else:
    cell_var_arrays_plot = cell_var_arrays
    cell_var_names_plot = cell_var_names

print(f'Will plot {len(cell_var_names_plot)} variable(s)')

Will plot 1 variable(s)


In [28]:
# Plot variables
elements = None
leg_sc = None
cb = None
leg_wt2 = None
bnd_flux = None
bnd_flux2 = None

for nplot, cell_var_name, cell_var_array in zip(itertools.count(),
                                                 cell_var_names_plot,
                                                 cell_var_arrays_plot):

    print(f'Plotting variable: {cell_var_name}')

    for vdim in range(len(cell_var_array.shape)):
        if len(cell_var_array.shape) > 1:
            # vector variable
            vtype = 'vector'
        else:
            vtype = 'scalar'

        if vtype == 'scalar':
            # cell values
            cell_avg = cell_var_array
        else:
            cell_avg = cell_var_array[:, vdim]

        if show_elements is True:
            # add element values:
            if nplot == 0:
                elements = matplotlib.collections.PatchCollection(
                    patches,
                    linewidths=0.0,
                    antialiased=False)
                ax.add_collection(elements)

            if elements is not None:
                elements.set_array(cell_avg)
                elements.set_clim(cell_var_array.min(), cell_var_array.max())
                elements.set_edgecolor('None')
                elements.set_zorder(0.1)
                elements.set_antialiased(True)
                elements.set_cmap(cmap)

        else:
            if vtype == 'scalar':
                if nplot == 0:
                    scatter_int = 1
                    leg_sc = ax.scatter(element_x[::scatter_int],
                                        element_y[::scatter_int],
                                        c=cell_var_array[::scatter_int],
                                        edgecolor='None', s=7)

                else:
                    if leg_sc is not None:
                        leg_sc.set_array(cell_var_array[::scatter_int])
                        if add_colorbar is True and cb is not None:
                            cb.set_clim(cell_var_array.min(),
                                        cell_var_array.max())

        xlim = [element_x.min(), element_x.max()]
        if ylim_fixed is None:
            ylim = [element_y.min(), element_y.max()]
        else:
            ylim = ylim_fixed

        if nplot == 0:
            # Plot sea level if applicable
            ax.plot([element_x.min(), 0], [0, 0],
                    color=sea_lvl_color, lw=0.25)

        if nplot == 0:
            va = (qx ** 2 + qy ** 2) ** 0.5
            scale = np.abs(va).max() * 10.0

            print('Converting flux to regular grid')
            xg, yg, qxg, qyg = convert_to_grid(element_x, element_y,
                                               qx * year,
                                               qy * year, dx=dx, dy=dy)

            mask = np.zeros_like(qxg, dtype=bool)
            mask[np.isnan(qxg)] = True

            qxgm = np.ma.array(qxg, mask=mask)
            qygm = np.ma.array(qyg, mask=mask)
            vag = (qxgm ** 2 + qygm ** 2) ** 0.5
            lw = lw_min + lw_max * vag / vag.max()

            leg_splot = ax.streamplot(xg, yg, qxgm, qygm,
                                      density=streamline_density,
                                      color='k', linewidth=lw)

            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
            ax.set_xlabel('Distance (m)')
            ax.set_ylabel('Elevation (m)')

        # Continue plotting boundary flux
        if nplot == 0:
            y0 = np.zeros_like(top_xy_sorted[:, 0])

            bnd_flux = \
                tax.fill_between(top_xy_sorted[:, 0],
                                 y0,
                                 nodal_flux_corrected[:, 1],
                                 where=nodal_flux_corrected[:, 1] > 0,
                                 facecolor='lightblue',
                                 edgecolor='black',
                                 lw=0.5)
            bnd_flux2 = \
                tax.fill_between(top_xy_sorted[:, 0],
                                 y0,
                                 nodal_flux_corrected[:, 1],
                                 where=nodal_flux_corrected[:, 1] < 0,
                                 facecolor='blue',
                                 edgecolor='black',
                                 lw=0.5)

            tax.set_xlim(xlim)
            tax.axhline(y=0, color='black', lw=0.5)
            tax.axvline(x=0, color='black', lw=0.5)
            tax.set_ylabel('Boundary flux (m/yr)')

            if add_title is True and plot_title is not None:
                tax.set_title(plot_title, fontsize='small')

            if tax_ylim_fixed is not None:
                tax.set_ylim(tax_ylim_fixed)
            else:
                tax.set_ylim(nodal_flux_corrected[:, 1].min() * 1.25,
                             nodal_flux_corrected[:, 1].max() * 1.25)

            if add_colorbar is True and cax2 is not None:
                if show_elements is True:
                    cb = fig.colorbar(elements,
                                      cax=cax2,
                                      orientation='horizontal')
                else:
                    if leg_sc is not None:
                        cb = fig.colorbar(leg_sc,
                                          cax=cax2,
                                          orientation='horizontal')
                if cb is not None:
                    cb.ax.tick_params(labelsize=leg_fontsize)

                    tick_locator = ticker.MaxNLocator(nbins=3)
                    cb.locator = tick_locator
                    cb.update_ticks()

        # Set colorbar labels
        if add_colorbar is True and cb is not None:
            if 'concentration' in cell_var_name:
                cb.set_label('Solute concentration (kg/kg)', fontsize='small')
            else:
                cb.set_label(cell_var_name)

        if nplot == 0 and h_top is not None:
            leg_wt2, = ax.plot(top_xy_sorted[:, 0], h_top,
                               color='gray', lw=1.5)

            # add vertical line at coastline
            ax.axvline(x=0, color='black', lw=0.5, zorder=0)

        if nplot == 0 and add_legend is True:
            if bnd_flux2 is not None and bnd_flux is not None:
                if leg_wt2 is not None:
                    legs = [bnd_flux2, bnd_flux, leg_wt2]
                    labels = ['recharge', 'discharge', 'watertable']
                else:
                    legs = [bnd_flux2, bnd_flux]
                    labels = ['recharge', 'discharge']
                tax.legend(legs, labels, frameon=False, loc='upper right', fontsize=leg_fontsize, ncol=3)

        # save regular figure:
        file_basename = ''.join(vtk_filename.split('.')[:-1])
        if vtype == 'scalar':
            fig_fn = file_basename + '_' + cell_var_name + figure_type
        else:
            fig_fn = file_basename + '_' + cell_var_name \
                     + ['x', 'y', 'z'][vdim] + figure_type

        fn_out = os.path.join(fig_folder, fig_fn)
        print(f'Saving figure: {fn_out}')
        fig.savefig(fn_out, dpi=dpi)

print('Done processing file')

pl.clf()
print('All done!')

Plotting variable: concentration
Converting flux to regular grid
Saving figure: model_output/salt_wedge_benchmark/fig/runS1_specified_pressure_[0, 19591]_final_output_concentration.png
Done processing file
All done!
